# Week 8, Day 5: Aliasing and copying

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w008d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
original = [1, 2, 3]
backup = original
original.append(4)
print(backup)

In [ ]:
original = [1, 2, 3]
backup = original
original.append(4)
backup[0] = 99
print(original)
print(backup)

> **Key idea.** Assignment never copies. After `b = a`, both names point at the same list, and any change to the list (through either name) is seen through both.

In [ ]:
def add_item(basket, item):
    basket.append(item)
    return basket

mine = ["rice"]
result = add_item(mine, "beans")
result.append("salt")
print(mine)
print(result is mine)

In [ ]:
a = [1, 2, 3]
b = a
c = [1, 2, 3]
print(a == b, a is b)
print(a == c, a is c)

In [ ]:
a = [1, 2, 3]
b = a
c = [1, 2, 3]
print(id(a) == id(b), id(a) == id(c))

> **Tip.** Use `==` to compare values, almost always. Use `is` only to ask "is this the very same object?", and to check for `None` (`x is None`), since there's only one `None`.

In [ ]:
original = [1, 2, 3]
a = original.copy()
b = list(original)
c = original[:]
original.append(4)
print(original, a, b, c)
print(a == [1, 2, 3], a is original)

In [ ]:
def median(values):
    ordered = values.copy()
    ordered.sort()
    mid = len(ordered) // 2
    if len(ordered) % 2:
        return ordered[mid]
    return (ordered[mid - 1] + ordered[mid]) / 2

data = [7, 1, 5, 3]
print(median(data))
print(data)

In [ ]:
grid = [[1, 2], [3, 4]]
copy_ = grid.copy()
copy_.append([5, 6])
copy_[0][0] = 99
print(grid)
print(copy_)

In [ ]:
import copy
grid = [[1, 2], [3, 4]]
deep = copy.deepcopy(grid)
deep[0][0] = 99
print(grid)
print(deep)

> **Key idea.** A **shallow** copy makes a new outer list but shares the items. A **deep** copy (`copy.deepcopy`) copies the items too, all the way down. For a list of numbers or strings, shallow is enough; for a list of lists you'll change, you need deep.

In [ ]:
grid = [[0] * 3] * 3
grid[0][0] = 1
print(grid)

In [ ]:
grid = []
for _ in range(3):
    grid.append([0] * 3)
grid[0][0] = 1
print(grid)

In [ ]:
def add_item(item, basket=[]):
    basket.append(item)
    return basket

print(add_item("rice"))
print(add_item("beans"))

In [ ]:
def add_item(item, basket=None):
    if basket is None:
        basket = []
    basket.append(item)
    return basket

print(add_item("rice"))
print(add_item("beans"))

### Exercise: Alias or not?

What does each block print?

```python
# A
x = [1, 2]
y = x
y.append(3)
print(x)

# B
x = [1, 2]
y = x[:]
y.append(3)
print(x)

# C
x = [[1], [2]]
y = x.copy()
y[0].append(9)
print(x)

# D
x = [1, 2]
y = x
x = [7]
print(y)
```

In [ ]:
print_a = ___
print_b = ___
print_c = ___
print_d = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each block\"\n    assert print_a == [1, 2, 3], \"y is an alias.\"\n    assert print_b == [1, 2], \"x[:] is a copy.\"\n    assert print_c == [[1, 9], [2]], \"A shallow copy shares the inner lists.\"\n    assert print_d == [1, 2], \"x = [7] only moves x.\"")

### Exercise: Don't touch my list

`top_three(scores)` should return the three highest scores, highest first, **without** changing the caller's list. Write it using a copy and `sort`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several lists\"\n    data = [50, 90, 70, 80, 60]\n    assert top_three(data) == [90, 80, 70]\n    assert data == [50, 90, 70, 80, 60], \"The caller's list must be unchanged.\"\n    assert top_three([1, 2]) == [2, 1]")

### Exercise: Equal or identical?

Fill in `True` or `False` for each.

```python
a = [1, 2]
b = [1, 2]
c = a
d = a[:]
```

In [ ]:
a_eq_b = ___   # a == b
a_is_b = ___   # a is b
a_is_c = ___   # a is c
a_is_d = ___   # a is d
a_eq_d = ___   # a == d

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each comparison\"\n    assert a_eq_b is True and a_is_b is False\n    assert a_is_c is True\n    assert a_is_d is False and a_eq_d is True")

### Exercise: A proper grid

Write `make_grid(rows, cols, fill=0)` that returns a grid (a list of `rows` lists, each of `cols` items equal to `fill`) where every row is a **separate** list.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_shape():\n    \"the right shape\"\n    g = make_grid(2, 3)\n    assert g == [[0, 0, 0], [0, 0, 0]]\n    assert make_grid(1, 2, \".\") == [[\".\", \".\"]]\ndef test_separate():\n    \"rows are separate lists\"\n    g = make_grid(3, 3)\n    g[0][0] = 1\n    assert g[1][0] == 0 and g[2][0] == 0, \"Changing one row changed another: build each row separately.\"")

### Exercise: A safe copy of a board

`board` is a list of lists. Make `saved` a copy of it that stays the same when `board` is changed afterwards, including changes to its inner lists.

In [ ]:
import copy
board = [["X", "O", " "], [" ", "X", " "], [" ", " ", "O"]]
saved = ___
board[0][2] = "X"
board.append(["!"])
print(saved)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"saved is unchanged\"\n    expect_output(\"[['X', 'O', ' '], [' ', 'X', ' '], [' ', ' ', 'O']]\")")

### Bug hunt: The backup changed

This should keep a backup of the list before clearing it, then print the backup, but it prints `[]`. Fix it.

In [ ]:
items = ["a", "b", "c"]
backup = items
items.clear()
print(backup)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the backup survives\"\n    expect_output(\"['a', 'b', 'c']\")")

### Bug hunt: Every row at once

This should put an `X` in the top-left corner only, but the whole first column fills up. Fix how the grid is made.

In [ ]:
grid = [["."] * 3] * 3
grid[0][0] = "X"
for row in grid:
    print("".join(row))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"only one X\"\n    expect_output(\"X..\\n...\\n...\")")

### Bug hunt: Items from nowhere

Each call should return a list with just the new item, unless a list is passed in. But the second call returns two items. Fix the function.

In [ ]:
def collect(item, into=[]):
    into.append(item)
    return into

print(collect(1))
print(collect(2))
print(collect(3, [0]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fresh lists\"\n    expect_output(\"[1]\\n[2]\\n[0, 3]\")")

### Puzzle: Change a copy

Arrange the lines to change a copy of a list, leaving the original alone, and print both. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Trace the objects

How many separate **list objects** exist at the end of each block? Count every list, including inner ones.

```python
# A
a = [1, 2]
b = a
c = a[:]

# B
row = [0, 0]
grid = [row, row]

# C
grid = [[0, 0], [0, 0]]
copy_ = grid.copy()
```

In [ ]:
lists_a = ___
lists_b = ___
lists_c = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each block\"\n    assert lists_a == 2, \"a and b are one list; c is a second.\"\n    assert lists_b == 2, \"row, and grid, which holds row twice.\"\n    assert lists_c == 4, \"Two inner lists, the original outer list, and the copy's outer list.\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a game with history

Noughts and crosses is played on a board kept as a list of 9 strings, positions 0 to 8, each `" "`, `"X"` or `"O"`.

Write `play(moves)` where `moves` is a list of positions. Players take turns, `X` first. The function returns a list of **snapshots**: the board as it was after each move, each one an independent list. A move to a position that's already taken is skipped (no snapshot, and the same player tries the next move). The original empty board must not appear in the result.

Then `show(board)` returns the board as three lines like `X|O| ` joined with `"\n"`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_snapshots():\n    \"independent snapshots\"\n    snaps = play([4, 0, 8])\n    assert len(snaps) == 3\n    assert snaps[0] == [\" \", \" \", \" \", \" \", \"X\", \" \", \" \", \" \", \" \"]\n    assert snaps[1] == [\"O\", \" \", \" \", \" \", \"X\", \" \", \" \", \" \", \" \"]\n    assert snaps[2] == [\"O\", \" \", \" \", \" \", \"X\", \" \", \" \", \" \", \"X\"]\n    snaps[0][0] = \"!\"\n    assert snaps[1][0] == \"O\", \"Each snapshot must be a separate list.\"\ndef test_taken():\n    \"a taken square is skipped\"\n    snaps = play([0, 0, 1])\n    assert len(snaps) == 2\n    assert snaps[1][:2] == [\"X\", \"O\"], \"The second 0 is skipped, so O plays at 1.\"\ndef test_show():\n    \"show\"\n    assert show([\"X\", \"O\", \" \", \" \", \"X\", \" \", \" \", \" \", \"O\"]) == \"X|O| \\n |X| \\n | |O\"")

## Warm-up

`backup` changed, although you never touched it. If that surprises you, today will put it right for good. If it doesn't, today will show you every place this matters.

## Aliases

`backup = original` doesn't copy anything. Assignment never copies: it ties a second label to the **same** object. The two names are **aliases**: two names for one list.

Step through it and watch the objects panel. There is only ever **one** list, with two arrows pointing at it. Every change, through either name, happens to that one list.

With numbers and strings, aliasing never showed, because they can't be changed: the only way to "change" one is to make a new object and move a name, which leaves the other name alone. With lists, mutation changes the shared object itself.

### Aliases you didn't write

Aliasing also happens without an `=` in sight:

- **Function parameters**: inside `f(items)`, the parameter is another name for the caller's list. (That's why the functions on Day 2 could change their arguments.)
- **Return values**: a function that returns a list it was given hands back the same object, not a copy.
- **Items of other lists**: a list can hold another list, and the same inner list can be held in several places.

## == versus is

There are two different questions you can ask about two lists:

- `a == b`: do they have **equal contents**?
- `a is b`: are they the **same object**?

`a` and `c` look identical, but they're two separate lists that happen to hold the same values. Changing one won't change the other. `id(x)` shows an object's identity number, so `a is b` is the same as `id(a) == id(b)`:

## Copies

When you want a separate list with the same items, make a **copy**. There are three common ways, all equivalent:

`original[:]` is a slice of the whole list, and every slice is a new list. Now each copy is independent: changing the original doesn't affect them.

### When do you need a copy?

When you want to change a list but keep the old version, or when a function shouldn't change the list it was given:

(Here `sorted(values)` would also do: it always makes a new list.)

### Shallow and deep

A list can hold other lists. A copy made with `.copy()`, `list()` or `[:]` is **shallow**: it's a new outer list, but its slots point at the **same** inner objects as the original.

Appending to the copy didn't touch the original (two separate outer lists). But `copy_[0][0] = 99` changed an **inner** list, and that inner list is shared, so the original sees it too. X-ray this one: you'll see two outer lists whose first two slots point at the same two inner lists.

To copy everything, all the way down, use `deepcopy` from the `copy` module:

## Two classic traps

### Trap 1: repeating a list of lists

`[0] * 3` makes a list of three zeros. So surely `[[0] * 3] * 3` makes a 3 by 3 grid?

Every row changed! `* 3` repeats the **reference** to the inner list, so all three rows are the same list. X-ray it: three arrows to one inner list.

Build each row separately instead:

`[0] * 3` is fine (numbers can't be changed, so sharing them doesn't matter). It's repeating a **list** that causes the trouble.

### Trap 2: a list as a default argument

Week 7 warned about this. Now you can see exactly why it goes wrong:

The default `[]` is made **once**, when `def` runs. Every call that doesn't pass a basket gets an alias for that same list, so items pile up between calls. The fix is the `None` pattern:

## Practice

## Quiz

## Daily challenge

## Before you go

Draw the names-and-objects picture for `grid = [[0] * 2] * 2`, and then for the loop that builds the grid properly. Then explain in one sentence each: aliasing, a shallow copy, a deep copy. Tomorrow is the Forge: the receipt maker from Week 5, rewritten with lists, and the weekly quiz.